In [11]:
print("hello")

hello


In [12]:
from pathlib import Path

working_directory = Path.cwd()
requirements_candidates = [
    candidate
    for parent in (working_directory, *working_directory.parents)
    for candidate in (parent / "requirements.txt", parent / "Openai-demo" / "requirements.txt")
]
requirements_path = next((path for path in requirements_candidates if path.is_file()), None)
if requirements_path is None:
    raise FileNotFoundError("Could not find the Openai-demo requirements.txt file.")

%pip install -r "{requirements_path}"
import sys
loaded_openai = sys.modules.get("openai")
if loaded_openai is not None and not hasattr(loaded_openai, "OpenAI"):
    raise RuntimeError("The OpenAI SDK was upgraded. Restart the notebook kernel, then run all cells again.")

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import (
    AuthenticationError,
    NotFoundError,
    OpenAI,
    RateLimitError,
)

In [14]:
from pathlib import Path

working_directory = Path.cwd()
dotenv_candidates = [
    candidate
    for parent in (working_directory, *working_directory.parents)
    for candidate in (parent / ".env", parent / "Openai-demo" / ".env")
]
dotenv_path = next((path for path in dotenv_candidates if path.is_file()), None)
if dotenv_path is None:
    raise FileNotFoundError("Could not find the Openai-demo .env file.")

load_dotenv(dotenv_path, override=True)
if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError(f"Set OPENAI_API_KEY in {dotenv_path}.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip()
client = OpenAI()

def run_openai_request(request):
    try:
        return request()
    except AuthenticationError as exc:
        raise RuntimeError(
            f"OpenAI rejected the key in {dotenv_path}. Replace it with a valid API key."
        ) from exc
    except NotFoundError as exc:
        raise RuntimeError(
            f"Model {MODEL!r} is unavailable. Set OPENAI_MODEL to a model enabled for your account."
        ) from exc
    except RateLimitError as exc:
        if "no credits remaining" in str(exc).lower():
            raise RuntimeError(
                "OpenAI authenticated successfully, but this project has no API credits remaining. "
                "Add credits in the OpenAI billing settings, then rerun the failed cells."
            ) from exc
        raise RuntimeError(
            f"OpenAI rate limit or quota error: {exc}"
        ) from exc

def create_chat_completion(messages, **kwargs):
    return run_openai_request(
        lambda: client.chat.completions.create(
            model=MODEL, messages=messages, **kwargs
        )
    )

def create_response(input_text):
    return run_openai_request(
        lambda: client.responses.create(model=MODEL, input=input_text)
    )

print(f"Loaded OpenAI settings from {dotenv_path}; using model {MODEL}.")

Loaded OpenAI settings from f:\Gen AI\OpenAI\Openai-demo\.env; using model gpt-4o-mini.


In [15]:
# The client reads OPENAI_API_KEY from the project .env loaded above.

In [16]:
# pandas is imported in the setup cell for displaying the model list.

In [17]:
try:
    models = client.models.list()
except AuthenticationError as exc:
    raise RuntimeError(
        f"OpenAI rejected the key in {dotenv_path}. Replace it with a valid API key."
    ) from exc

pd.DataFrame([model.model_dump() for model in models.data])

,id,created,object,owned_by,shutdown_date
0,gpt-realtime-2.1-mini,1782254706,model,system,NaN
1,gpt-realtime-2025-08-28,1756271773,model,system,2027-01-20
2,tts-1-hd,1699046015,model,system,2027-01-06
3,gpt-realtime-1.5,1771461469,model,system,NaN
4,gpt-realtime,1756271701,model,system,2027-01-20
...,...,...,...,...,...
91,gpt-4.1-mini-2025-04-14,1744317547,model,system,2027-01-14
92,gpt-image-2.5-flare,1788563147,model,system,NaN
93,gpt-6.1-terra,1791908780,model,system,NaN
94,gpt-6-sol,1789405937,model,system,NaN


# ChatCompletion API and Completion API

- Completion API: Hooks you up with text completions from a single prompt. 

- Chat Completion API: Nails it in the chat game, keeping the conversational flow intact

# ChatCompletion API 

In [ ]:
prompt = "Hello, how are you?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
)

print(response.choices[0].message.content)


In [ ]:
prompt = "Hello, how are you?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
)

print(response)


In [ ]:
prompt1 = "Hello, how are you?"
prompt2 = "I am 25 years old and I am a programmer."
prompt3 = "Tell me about me."

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt1},
        {"role": "user", "content": prompt2},
        {"role": "user", "content": prompt3},
    ]
)

print(response.choices[0].message.content)


In [ ]:
prompt = "What is Python?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_completion_tokens=25,
)

print(response.choices[0].message.content)


In [ ]:
prompt = "What is Python?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_completion_tokens=25,
)

print(response)


In [ ]:
prompt = "What is Python?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)


In [ ]:
prompt = "What is Python?"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
    n=3,
)

print(response)


In [ ]:
prompt = "Give me the sentiment of this sentence: This movie is amazing!"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)


In [ ]:
prompt = "Translate this sentence into Hindi: This movie is amazing!"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)


In [ ]:
prompt = "Detect the language of this sentence: This movie is amazing!"

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)


In [ ]:
prompt = "Give me Python code to add two numbers."

response = create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)


# Responses API (modern replacement for the legacy Completion API)

In [ ]:
prompt = "What is Python?"
response = create_response(prompt)
print(response.output_text)